# Zadanie 4: Wizualizacja danych i budowanie interaktywnego dashboardu
## Analiza rynku koncertowego w Polsce

### Generowanie danych
Uruchamiamy kod do wygenerowania pliku `koncerty_polska.csv` zawierającego dane o koncertach.

In [1]:
# Kod generujący dane wejściowe
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
n = 1200

miasta = {
    "Warszawa":   (52.2297, 21.0122, 1.00),
    "Kraków":     (50.0647, 19.9450, 0.75),
    "Wrocław":    (51.1079, 17.0385, 0.65),
    "Poznań":     (52.4064, 16.9252, 0.55),
    "Gdańsk":     (54.3520, 18.6466, 0.55),
    "Łódź":       (51.7592, 19.4560, 0.50),
    "Katowice":   (50.2649, 19.0238, 0.45),
    "Lublin":     (51.2465, 22.5684, 0.30),
    "Białystok":  (53.1325, 23.1688, 0.25),
    "Szczecin":   (53.4285, 14.5528, 0.35),
}

gatunki = ["rock", "pop", "hip-hop", "electronic", "jazz",
           "classical", "folk", "metal", "indie", "reggae"]

typy_obiektow = ["klub", "arena", "stadion", "festiwal", "teatr", "amfiteatr"]
kapacjety = {
    "klub": (200, 1500), "arena": (3000, 15000), "stadion": (20000, 70000),
    "festiwal": (10000, 80000), "teatr": (400, 2000), "amfiteatr": (1500, 8000),
}
cena_bazowa = {
    "rock": 150, "pop": 200, "hip-hop": 180, "electronic": 160, "jazz": 130,
    "classical": 110, "folk": 90, "metal": 140, "indie": 100, "reggae": 110,
}
cena_mnoznik = {
    "klub": 0.7, "arena": 1.3, "stadion": 1.8,
    "festiwal": 1.5, "teatr": 1.2, "amfiteatr": 1.0,
}

start_date = datetime(2024, 1, 1)
daty = [start_date + timedelta(days=int(d)) for d in np.random.randint(0, 730, n)]

wagi = np.array([miasta[m][2] for m in miasta])
miasto = np.random.choice(list(miasta.keys()), n, p=wagi / wagi.sum())
gatunek = np.random.choice(gatunki, n)
typ_obiektu = np.random.choice(typy_obiektow, n,
                                p=[0.40, 0.15, 0.05, 0.10, 0.15, 0.15])

pojemnosc = np.array([np.random.randint(*kapacjety[t]) for t in typ_obiektu])
wypelnienie = np.clip(np.random.beta(5, 2, n), 0.15, 1.0)
sprzedane = (pojemnosc * wypelnienie).astype(int)

cena = np.array([cena_bazowa[g] * cena_mnoznik[t] for g, t in zip(gatunek, typ_obiektu)])
cena = np.round(cena * np.random.uniform(0.7, 1.4, n), -1)
przychod = (cena * sprzedane).astype(int)

df = pd.DataFrame({
    "event_id": range(50001, 50001 + n),
    "data": daty,
    "miasto": miasto,
    "latitude": [miasta[m][0] for m in miasto],
    "longitude": [miasta[m][1] for m in miasto],
    "gatunek": gatunek,
    "typ_obiektu": typ_obiektu,
    "pojemnosc": pojemnosc,
    "bilety_sprzedane": sprzedane,
    "cena_biletu_pln": cena,
    "przychod_pln": przychod,
})

df.to_csv("koncerty_polska.csv", index=False)
print(f"Wygenerowano plik 'koncerty_polska.csv' zawierający {len(df)} rekordów.")

Wygenerowano plik 'koncerty_polska.csv' zawierający 1200 rekordów.


## Część 1 — Wczytanie i wstępna eksploracja danych (1 pkt)
Wczytujemy dane z pliku CSV, definiujemy typ danych dat jako datetime oraz wyświetlamy podstawowe metadane (wymiary, podgląd danych, typy kolumn, unikalne miasta i gatunki).

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

def prepare_data(data_df: pd.DataFrame) -> pd.DataFrame:
    """Przygotowuje ramkę danych do dalszej analizy i wizualizacji."""
    df_copy = data_df.copy()
    # Konwersja kolumny data na datetime, jeśli jeszcze nie jest
    df_copy["data"] = pd.to_datetime(df_copy["data"])
    # Dodanie kolumny rok-miesiąc
    df_copy["rok_miesiac"] = df_copy["data"].dt.to_period("M").astype(str)
    # Dodanie kolumny wypelnienie
    df_copy["wypelnienie"] = df_copy["bilety_sprzedane"] / df_copy["pojemnosc"]
    return df_copy

# Wczytanie danych z parsowaniem dat
df_raw = pd.read_csv("koncerty_polska.csv", parse_dates=["data"])
df = prepare_data(df_raw)

# Podstawowe sprawdzenie struktury
print(f"Kształt zbioru danych: {df.shape}")
print("\nTypy kolumn:")
print(df.dtypes)

# Sprawdzenie unikalnych kategorii
print(f"\nLiczba unikalnych miast: {df['miasto'].nunique()} {df['miasto'].unique().tolist()}")
print(f"Liczba unikalnych gatunków: {df['gatunek'].nunique()} {df['gatunek'].unique().tolist()}")

Kształt zbioru danych: (1200, 11)

Typy kolumn:
event_id                     int64
data                datetime64[ns]
miasto                      object
latitude                   float64
longitude                  float64
gatunek                     object
typ_obiektu                 object
pojemnosc                    int64
bilety_sprzedane             int64
cena_biletu_pln            float64
przychod_pln                 int32
dtype: object

Liczba unikalnych miast: 10 ['Kraków', 'Szczecin', 'Poznań', 'Katowice', 'Warszawa', 'Wrocław', 'Łódź', 'Gdańsk', 'Lublin', 'Białystok']
Liczba unikalnych gatunków: 10 ['indie', 'metal', 'jazz', 'pop', 'folk', 'classical', 'electronic', 'rock', 'hip-hop', 'reggae']


## Część 2 — Wykres słupkowy (2 pkt)
Obliczamy łączny przychód ze sprzedaży biletów w podziale na miasta. Zgodnie z zaleceniami, kod wykresu opakowujemy w osobną funkcję zwracającą obiekt `go.Figure`.

In [3]:
def plot_revenue_by_city(data_df: pd.DataFrame) -> go.Figure:
    """Tworzy wykres słupkowy łącznego przychodu według miast."""
    # Agregacja przychodu według miast i sortowanie malejąco
    df_przychody = data_df.groupby("miasto")["przychod_pln"].sum().reset_index()
    df_przychody = df_przychody.sort_values(by="przychod_pln", ascending=False)

    # Generowanie wykresu słupkowego
    fig = px.bar(
        df_przychody,
        x="miasto",
        y="przychod_pln",
        title="Łączny przychód z koncertów w polskich miastach (2024-2025)",
        labels={
            "miasto": "Miasto",
            "przychod_pln": "Łączny przychód (PLN)"
        },
        color="przychod_pln",
        color_continuous_scale="Viridis",
        template="plotly_white"
)

    # Układ wykresu
    fig.update_layout(
        xaxis_title="Miasto",
        yaxis_title="Przychód [PLN]",
        coloraxis_showscale=False,
        width=900,
        height=500
    )
    fig.update_traces(
        hovertemplate="<b>%{x}</b><br>Przychód: %{y:,.0f} PLN<extra></extra>"
    )
    return fig

# Wyświetlenie wykresu
fig_bar = plot_revenue_by_city(df)
fig_bar.show()

## Część 3 — Wykresy liniowe (szereg czasowy) (2 pkt)
Tworzymy funkcje generujące wykresy czasowe: ogólną miesięczną liczbę koncertów oraz liczbę koncertów w podziale na typy obiektów.

In [4]:
def plot_monthly_concerts(data_df: pd.DataFrame) -> go.Figure:
    """Tworzy wykres liniowy przedstawiający miesięczną liczbę koncertów."""
    df_miesiecznie = data_df.groupby("rok_miesiac").size().reset_index(name="liczba_koncertow")
    fig = px.line(
        df_miesiecznie,
        x="rok_miesiac",
        y="liczba_koncertow",
        title="Miesięczna liczba koncertów w Polsce",
        labels={
            "rok_miesiac": "Miesiąc",
            "liczba_koncertow": "Liczba koncertów"
        },
        markers=True,
        template="plotly_white"
)
    fig.update_layout(width=900, height=500)
    fig.update_traces(
        line=dict(color="royalblue", width=3),
        hovertemplate="<b>%{x}</b><br>Liczba koncertów: %{y}<extra></extra>"
    )
    return fig

def plot_monthly_concerts_by_venue(data_df: pd.DataFrame) -> go.Figure:
    """Tworzy wykres liniowy przedstawiający miesięczną liczbę koncertów z podziałem na typy obiektów."""
    df_miesiecznie_obiekty = data_df.groupby(["rok_miesiac", "typ_obiektu"]).size().reset_index(name="liczba_koncertow")
    fig = px.line(
        df_miesiecznie_obiekty,
        x="rok_miesiac",
        y="liczba_koncertow",
        color="typ_obiektu",
        title="Miesięczna liczba koncertów w podziale na typy obiektów",
        labels={
            "rok_miesiac": "Miesiąc",
            "liczba_koncertow": "Liczba koncertów",
            "typ_obiektu": "Typ obiektu"
        },
        markers=True,
        template="plotly_white"
)
    fig.update_layout(width=900, height=500)
    fig.update_traces(
        hovertemplate="<b>%{x}</b> (%{legendgroup})<br>Liczba koncertów: %{y}<extra></extra>"
    )
    return fig

# Wyświetlenie wykresów
fig_line1 = plot_monthly_concerts(df)
fig_line1.show()

fig_line2 = plot_monthly_concerts_by_venue(df)
fig_line2.show()

## Część 4 — Histogram i boxplot (2 pkt)
Definiujemy funkcje dla histogramu cen biletów oraz wykresu pudełkowego przychodów w zależności od typu obiektu.

In [5]:
def plot_ticket_price_histogram(data_df: pd.DataFrame) -> go.Figure:
    """Tworzy histogram cen biletów."""
    fig = px.histogram(
        data_df,
        x="cena_biletu_pln",
        nbins=50,
        title="Rozkład cen biletów na koncerty (nbins=50)",
        labels={
            "cena_biletu_pln": "Cena biletu (PLN)",
            "count": "Liczba koncertów"
        },
        color_discrete_sequence=["indianred"],
        template="plotly_white"
    )
    fig.update_layout(
        yaxis_title="Liczba koncertów",
        width=900,
        height=500
    )
    return fig

def plot_revenue_by_venue_boxplot(data_df: pd.DataFrame) -> go.Figure:
    """Tworzy boxplot przychodów według typu obiektu."""
    fig = px.box(
        data_df,
        x="typ_obiektu",
        y="przychod_pln",
        title="Przychody z koncertów w zależności od typu obiektu",
        labels={
            "typ_obiektu": "Typ obiektu",
            "przychod_pln": "Przychód z pojedynczego koncertu (PLN)"
        },
        color="typ_obiektu",
        color_discrete_sequence=px.colors.qualitative.Safe,
        template="plotly_white"
    )
    fig.update_layout(
        showlegend=False,
        width=900,
        height=500
    )
    return fig

# Wyświetlenie wykresów
fig_hist = plot_ticket_price_histogram(df)
fig_hist.show()

fig_box = plot_revenue_by_venue_boxplot(df)
fig_box.show()

### Komentarz do Części 4

Najwyższe przychody z pojedynczych koncertów generują **stadiony** oraz **festiwale**. 
Wynika to z ich dużej pojemności (sięgającej do kilkudziesięciu tysięcy widzów) oraz wyższego mnożnika ceny biletu (odpowiednio 1.8 dla stadionów i 1.5 dla festiwali). 
W przeciwieństwie do nich, kluby i teatry charakteryzują się znacznie mniejszymi przychodami jednostkowymi z powodu ograniczonej pojemności miejsc i niższych cen wejściówek.

## Część 5 — Wykres rozrzutu z kodowaniem parametrów (2 pkt)
Definiujemy funkcję do badania korelacji pomiędzy ceną a wskaźnikiem wypełnienia sali.

In [6]:
def plot_price_vs_occupancy_scatter(data_df: pd.DataFrame) -> go.Figure:
    """Tworzy wykres rozrzutu badający zależność ceny biletu od wskaźnika wypełnienia sali."""
    fig = px.scatter(
        data_df,
        x="cena_biletu_pln",
        y="wypelnienie",
        color="gatunek",
        size="pojemnosc",
        hover_data=["miasto", "typ_obiektu"],
        title="Zależność pomiędzy ceną biletu a wypełnieniem sali",
        labels={
            "cena_biletu_pln": "Cena biletu (PLN)",
            "wypelnienie": "Wskaźnik wypełnienia",
            "gatunek": "Gatunek muzyczny",
            "pojemnosc": "Pojemność obiektu"
        },
        template="plotly_white"
    )
    fig.update_layout(
        width=900,
        height=600
    )
    return fig

# Wyświetlenie wykresu
fig_scatter = plot_price_vs_occupancy_scatter(df)
fig_scatter.show()

### Komentarz do Części 5

**Analiza zależności między ceną biletu a wypełnieniem sali:**

Na wykresie nie obserwuje się zależności pomiędzy ceną wejściówki a stopniem zapełnienia sali. Punkty są rozmieszczone w sposób jednorodny, tworząc rozproszoną chmurę bez wyraźnego trendu. 

Brak korelacji wynika ze specyfiki modelu generowania danych. Wskaźnik wypełnienia został wylosowany przy użyciu rozkładu prawdopodobieństwa beta niezależnie od ceny, która była kalkulowana na podstawie parametrów gatunku i obiektu. W realnych warunkach rynkowych wysoka cena mogłaby wpływać ujemnie na popyt, jednak w tym przypadku popyt wykazuje pełną sztywność.

## Część 6 — Mapa geograficzna (2 pkt)
Definiujemy funkcję do rysowania mapy przedstawiającej lokalizacje koncertów w Polsce.

In [7]:
def plot_concert_map(data_df: pd.DataFrame) -> go.Figure:
    """Tworzy interaktywną mapę Polski prezentującą dane o koncertach."""
    # Agregacja danych do poziomu miast
    df_miasta_agg = data_df.groupby("miasto").agg(
        srednia_cena=("cena_biletu_pln", "mean"),
        liczba_koncertow=("event_id", "count"),
        laczny_przychod=("przychod_pln", "sum"),
        latitude=("latitude", "first"),
        longitude=("longitude", "first")
    ).reset_index()

    # Generowanie mapy
    fig = px.scatter_mapbox(
        df_miasta_agg,
        lat="latitude",
        lon="longitude",
        size="liczba_koncertow",
        color="srednia_cena",
        color_continuous_scale="RdBu_r",
        hover_name="miasto",
        hover_data={
            "latitude": False,
            "longitude": False,
            "liczba_koncertow": True,
            "srednia_cena": ":.2f",
            "laczny_przychod": ":,.0f"
        },
        zoom=5.2,
        center=dict(lat=52.0, lon=19.0),
        mapbox_style="open-street-map",
        title="Średnia cena biletu oraz liczba koncertów w miastach Polski",
        height=600
    )
    fig.update_layout(
        width=950,
        margin=dict(r=10, t=50, l=10, b=10)
    )
    return fig

# Wyświetlenie mapy
fig_map = plot_concert_map(df)
fig_map.show()

## Część 7 — Kompozycja: subploty (2 pkt)
Definiujemy funkcję łączącą cztery analizy w jeden wspólny mini-dashboard w układzie siatki 2x2.

In [8]:
def plot_concert_subplots_dashboard(data_df: pd.DataFrame) -> go.Figure:
    """Tworzy zintegrowaną figurę 2x2 z 4 różnymi analizami rynkowymi."""
    df_przychody_miasta = data_df.groupby("miasto")["przychod_pln"].sum().reset_index().sort_values("przychod_pln", ascending=False)
    df_gatunki_count = data_df.groupby("gatunek").size().reset_index(name="liczba_koncertow").sort_values("liczba_koncertow", ascending=False)

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Łączny przychód miast (PLN)",
            "Liczba koncertów według gatunku",
            "Rozkład cen biletów (PLN)",
            "Rozkład wskaźnika wypełnienia wg typu obiektu"
        )
    )

    # Wykres 1: Przychód wg miast
    fig.add_trace(
        go.Bar(
            x=df_przychody_miasta["miasto"],
            y=df_przychody_miasta["przychod_pln"],
            name="Przychód",
            marker_color="teal"
        ),
        row=1, col=1
    )

    # Wykres 2: Liczba koncertów wg gatunków
    fig.add_trace(
        go.Bar(
            x=df_gatunki_count["gatunek"],
            y=df_gatunki_count["liczba_koncertow"],
            name="Liczba koncertów",
            marker_color="orchid"
        ),
        row=1, col=2
    )

    # Wykres 3: Rozkład cen biletów
    fig.add_trace(
        go.Histogram(
            x=data_df["cena_biletu_pln"],
            nbinsx=40,
            name="Ceny biletów",
            marker_color="indianred"
        ),
        row=2, col=1
    )

    # Wykres 4: Rozkład wypełnienia wg typu obiektu
    for t in data_df["typ_obiektu"].unique():
        fig.add_trace(
            go.Box(
                y=data_df[data_df["typ_obiektu"] == t]["wypelnienie"],
                name=t,
                showlegend=False
            ),
            row=2, col=2
        )

    fig.update_layout(
        title_text="Podsumowanie rynku koncertów w Polsce — Panel 2x2",
        title_font_size=20,
        height=800,
        width=1000,
        template="plotly_white",
        showlegend=False
    )
    return fig

# Wyświetlenie siatki dashboardu
fig_sub = plot_concert_subplots_dashboard(df)
fig_sub.show()

## Część 8 — Podsumowanie i wnioski (1 pkt)
Główne spostrzeżenia z przeprowadzonej analizy rynku koncertowego w Polsce.

### Główne wnioski z analizy:

1. **Dominacja rynku warszawskiego**: Warszawa jest głównym ośrodkiem koncertowym w Polsce. Odpowiada za największą liczbę zorganizowanych wydarzeń oraz najwyższy łączny przychód finansowy. Znaczący udział w rynku mają także Kraków i Wrocław, podczas gdy mniejsze miasta (takie jak Białystok czy Lublin) stanowią margines działalności koncertowej.
2. **Brak sezonowości**: Z analizy szeregu czasowego wynika, że liczba koncertów utrzymuje się na stałym poziomie (około 50 na miesiąc). Wynika to bezpośrednio z założeń generatora danych, w którym daty zostały rozłożone równomiernie w okresie dwóch lat.
3. **Rentowność obiektów**: Najbardziej opłacalne pod względem finansowym są stadiony oraz festiwale. Kluby muzyczne i teatry generują zdecydowanie mniejsze przychody jednostkowe, co jest spowodowane mniejszą pojemnością widowni.
4. **Sztywność popytu**: Cena wejściówki nie wpływa na stopień napełnienia sal. Wskaźnik wypełnienia rozkłada się losowo niezależnie od ceny, co wskazuje na sztywność popytu w wygenerowanym modelu.
5. **Struktura cen biletów**: Rozkład cen biletów ma charakter wielomodalny. Powstanie kilku wierzchołków cenowych wiąże się ze zróżnicowaniem cen bazowych dla poszczególnych gatunków muzycznych oraz z zastosowaniem mnożników pojemności obiektów.